# Reproduce: `exp_adaptive_ratio/w30_h64_lr005_weighted`

**Axis:** lr 0.01→0.005 @ w30 h64 weighted e300

**Pins:** dataset=v2 seed=42 window=30.0s epochs=300 edge_weight=True scorer=stochastic

This notebook re-executes the same CLI as the original run into `nb_repro/` under this run dir, then compares primary medians/ratio to frozen `comparison.json` / `reproduce_config.json` expected values (tolerance for stochastic recon).

```bash
cd .
.venv/bin/python -m abrg.validate_reproduce --run-dir abrg/output/exp_adaptive_ratio/w30_h64_lr005_weighted
# or: jupyter nbconvert --to notebook --execute abrg/output/exp_adaptive_ratio/w30_h64_lr005_weighted/reproduce.ipynb
```


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

CWD = Path.cwd().resolve()
REPO_ROOT = CWD if (CWD / "abrg").is_dir() else CWD.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from abrg.reproduce import (
    cli_argv_from_config,
    compare_metrics,
    primary_arm_metrics,
)

RUN_DIR = (REPO_ROOT / 'abrg/output/exp_adaptive_ratio/w30_h64_lr005_weighted').resolve()
CFG = json.loads((RUN_DIR / "reproduce_config.json").read_text(encoding="utf-8"))
EXPECTED_JSON = RUN_DIR / "comparison.json"
print("RUN_DIR:", RUN_DIR)
print("axis:", CFG.get("axis"))
print("pins:", CFG.get("pins"))


In [ ]:
REPRO_DIR = RUN_DIR / "nb_repro"
if REPRO_DIR.exists():
    import shutil
    shutil.rmtree(REPRO_DIR)
REPRO_DIR.mkdir(parents=True)

argv = [sys.executable, *cli_argv_from_config(CFG, REPRO_DIR)]
print("Running:", " ".join(argv))
proc = subprocess.run(argv, cwd=REPO_ROOT)
print("exit_code:", proc.returncode)
assert proc.returncode == 0, "reproduce CLI failed"


In [ ]:
actual_cmp = json.loads((REPRO_DIR / "comparison.json").read_text(encoding="utf-8"))
actual = primary_arm_metrics(actual_cmp)
expected = CFG.get("expected") or primary_arm_metrics(
    json.loads(EXPECTED_JSON.read_text(encoding="utf-8"))
)
report = compare_metrics(
    expected,
    actual,
    atol_median=float(CFG.get("atol_median", 0.02)),
    atol_ratio=float(CFG.get("atol_ratio", 0.05)),
)
(RUN_DIR / "nb_validate_report.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(json.dumps(report, indent=2))
assert report["ok"], "reproduce metrics outside tolerance — see nb_validate_report.json"
print("REPRODUCE OK")
